# FluxAudio-S 150k 正式評估

本notebook使用已確認完整的測試資料：100首真實30秒MP3、300段10秒切片與300筆caption。

流程：

1. 驗證150k權重與T4。
2. 從53 GB ZIP中只抽取100首test MP3和3個manifest，不解壓整份訓練資料。
3. 依`partitions.tsv`建立300個真實10秒reference。
4. 用150k模型生成300個9.975秒樣本；每首完成後立即保存到Drive，可中斷續跑。
5. 計算matched／shuffled CLAP。
6. 檢查reference與generated沒有路徑或內容重複，再執行FAD。

請使用 **T4 GPU**。第4步耗時最久，重新執行會跳過已完成的有效音訊。


## 1. 掛載Drive、確認T4與設定路徑


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import subprocess
import time

MOUNT = Path("/content/drive")
if not (MOUNT / "MyDrive").exists():
    if MOUNT.exists() and any(MOUNT.iterdir()):
        backup = Path(f"/content/drive_local_backup_{int(time.time())}")
        MOUNT.rename(backup)
        print("未掛載時建立的本機資料已移到：", backup)
    MOUNT.mkdir(parents=True, exist_ok=True)
    drive.mount(str(MOUNT))
else:
    print("Drive已掛載")

if not shutil.which("nvidia-smi"):
    raise RuntimeError("未偵測到GPU；請將執行階段改成T4 GPU後重新執行")

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
    check=True,
)
print("GPU：", gpu.stdout.strip())

DRIVE_ROOT = Path("/content/drive/MyDrive")
REPO = Path("/content/ICME26-ATTM-GC-FluxAudio")
DATA_ZIP = DRIVE_ROOT / "FluxAudio_data/jamendo_meanaudio_ready_cache.zip"
WEIGHT_CACHE = DRIVE_ROOT / "FluxAudio_data/meanaudio_weights"

MODEL_DIR = DRIVE_ROOT / "FluxAudio_checkpoints/fluxaudio_s_150k_stage6"
MODEL_WEIGHTS = MODEL_DIR / "fluxaudio_s_150k_stage6_last.pth"

EVAL_ROOT = DRIVE_ROOT / "FluxAudio_evaluation/150k"
REFERENCE_DIR = EVAL_ROOT / "reference_300x10s"
GENERATED_DIR = EVAL_ROOT / "generated_300x9.975s"
RESULT_DIR = EVAL_ROOT / "results"

for folder in [EVAL_ROOT, REFERENCE_DIR, GENERATED_DIR, RESULT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

required = [DATA_ZIP, MODEL_WEIGHTS, WEIGHT_CACHE]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("缺少：\n" + "\n".join(str(path) for path in missing))

print("150k權重：", round(MODEL_WEIGHTS.stat().st_size / 1024**3, 2), "GB")
print("評估輸出：", EVAL_ROOT)


## 2. 建立推論環境與複製五個必要權重


In [ ]:
import sys

if REPO.exists() and not (REPO / "infer.py").exists():
    backup = Path(f"/content/FluxAudio_incomplete_{int(time.time())}")
    REPO.rename(backup)
    print("不完整專案已移到：", backup)

if not REPO.exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/ntu-musicailab/ICME26-ATTM-GC-FluxAudio.git",
            str(REPO),
        ],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)],
    check=True,
)

weight_names = [
    "v1-16.pth",
    "best_netG.pt",
    "empty_string_t5.pth",
    "empty_string_clap_c.pth",
    "music_speech_audioset_epoch_15_esc_89.98.pt",
]
target_weights = REPO / "weights"
target_weights.mkdir(parents=True, exist_ok=True)

for name in weight_names:
    source = WEIGHT_CACHE / name
    destination = target_weights / name
    if not source.exists():
        raise FileNotFoundError(source)
    if not destination.exists() or destination.stat().st_size != source.stat().st_size:
        print("複製：", name)
        shutil.copy2(source, destination)

help_result = subprocess.run(
    [sys.executable, "infer.py", "--help"],
    cwd=REPO,
    capture_output=True,
    text=True,
)
if help_result.returncode != 0:
    raise RuntimeError(help_result.stderr[-4000:])

print("純推論環境準備完成")


## 3. 從ZIP只抽取test manifests與100首真實MP3


In [ ]:
from zipfile import ZipFile

LOCAL_TEST = Path("/content/eval_150k_test")
RAW_AUDIO_DIR = LOCAL_TEST / "audios_real"
RAW_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

manifest_names = {
    "npz": "data/jamendo_meanaudio_ready/test/npz.tsv",
    "partitions": "data/jamendo_meanaudio_ready/test/partitions.tsv",
    "songs": "data/jamendo_meanaudio_ready/test/jamendo_test.tsv",
}
audio_prefix = "data/jamendo_meanaudio_ready/test/audios_real/"

with ZipFile(DATA_ZIP) as archive:
    archive_names = set(archive.namelist())
    for label, member in manifest_names.items():
        if member not in archive_names:
            raise FileNotFoundError(f"ZIP缺少 {member}")
        destination = LOCAL_TEST / Path(member).name
        if not destination.exists():
            with archive.open(member) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)

    audio_members = sorted(
        name for name in archive_names
        if name.startswith(audio_prefix) and name.lower().endswith(".mp3")
    )
    if len(audio_members) != 100:
        raise RuntimeError(f"預期100首test MP3，實際{len(audio_members)}")

    for index, member in enumerate(audio_members, 1):
        destination = RAW_AUDIO_DIR / Path(member).name
        if not destination.exists() or destination.stat().st_size == 0:
            with archive.open(member) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)
        if index % 20 == 0:
            print(f"已抽取 {index}/100")

print("真實MP3：", len(list(RAW_AUDIO_DIR.glob("*.mp3"))))
print("本機test資料：", LOCAL_TEST)


## 4. 建立300個真實10秒reference


In [ ]:
import csv

PARTITIONS_TSV = LOCAL_TEST / "partitions.tsv"
NPZ_TSV = LOCAL_TEST / "npz.tsv"
SAMPLE_RATE = 44100

with PARTITIONS_TSV.open(encoding="utf-8") as file:
    partitions = list(csv.DictReader(file, delimiter="\t"))

if len(partitions) != 300:
    raise RuntimeError(f"partitions應為300筆，實際{len(partitions)}")

for index, row in enumerate(partitions, 1):
    clip_id = row["id"]
    song_id = row["name"]
    source = RAW_AUDIO_DIR / f"{song_id}.mp3"
    destination = REFERENCE_DIR / f"{clip_id}.wav"

    if destination.exists() and destination.stat().st_size > 500_000:
        continue
    if not source.exists():
        raise FileNotFoundError(source)

    start_seconds = int(row["start_sample"]) / SAMPLE_RATE
    duration_seconds = (int(row["end_sample"]) - int(row["start_sample"])) / SAMPLE_RATE

    command = [
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-ss", f"{start_seconds:.6f}",
        "-i", str(source),
        "-t", f"{duration_seconds:.6f}",
        "-ar", "44100",
        "-ac", "2",
        "-c:a", "pcm_s16le",
        str(destination),
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"切分失敗 {clip_id}: {result.stderr[-2000:]}")

    if index % 25 == 0:
        print(f"reference完成 {index}/300")

reference_files = sorted(REFERENCE_DIR.glob("*.wav"))
if len(reference_files) != 300:
    raise RuntimeError(f"reference預期300，實際{len(reference_files)}")

print("Reference完成：", len(reference_files))
print("位置：", REFERENCE_DIR)


## 5. 生成150k的300個測試音訊

每個片段使用對應caption；同一首歌的`_0/_1/_2`分別使用seed 42/43/44。每完成一首立即複製到Drive。若執行階段中止，重新執行本格會跳過已完成檔案。


In [ ]:
import json

with NPZ_TSV.open(encoding="utf-8") as file:
    caption_rows = list(csv.DictReader(file, delimiter="\t"))

if len(caption_rows) != 300:
    raise RuntimeError(f"caption預期300，實際{len(caption_rows)}")

manifest = []

for index, row in enumerate(caption_rows, 1):
    clip_id = row["id"]
    caption = row["caption"]
    suffix = int(clip_id.rsplit("_", 1)[-1])
    seed = 42 + suffix
    destination = GENERATED_DIR / f"{clip_id}.wav"

    manifest.append({
        "id": clip_id,
        "caption": caption,
        "seed": seed,
        "generated": str(destination),
        "reference": str(REFERENCE_DIR / f"{clip_id}.wav"),
    })

    if destination.exists() and destination.stat().st_size > 100_000:
        if index % 10 == 0:
            print(f"已存在 {index}/300")
        continue

    temp_dir = Path("/content/eval_150k_generate") / clip_id
    temp_dir.mkdir(parents=True, exist_ok=True)

    command = [
        sys.executable, "infer.py",
        "--variant", "fluxaudio_s",
        "--model_path", str(MODEL_WEIGHTS),
        "--encoder_name", "t5_clap",
        "--use_rope",
        "--text_c_dim", "512",
        "--prompt", caption,
        "--duration", "9.975",
        "--cfg_strength", "4.5",
        "--num_steps", "25",
        "--seed", str(seed),
        "--output", str(temp_dir),
    ]

    result = subprocess.run(command, cwd=REPO, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-4000:])
        raise RuntimeError(f"生成失敗：{clip_id}")

    candidates = list(temp_dir.rglob("*.wav")) + list(temp_dir.rglob("*.flac"))
    if not candidates:
        raise FileNotFoundError(f"生成後找不到音訊：{clip_id}")

    generated = max(candidates, key=lambda path: path.stat().st_mtime)
    shutil.copy2(generated, destination)

    if index % 10 == 0:
        print(f"生成完成 {index}/300")

(EVAL_ROOT / "manifest_300.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

generated_files = sorted(GENERATED_DIR.glob("*.wav"))
print("生成音訊：", len(generated_files))
if len(generated_files) != 300:
    raise RuntimeError("尚未完成300首；可稍後重新執行本格續跑")

print("300首生成完成")


## 6. 評估前防呆：數量、路徑與內容雜湊


In [ ]:
import hashlib

reference_files = sorted(REFERENCE_DIR.glob("*.wav"))
generated_files = sorted(GENERATED_DIR.glob("*.wav"))

if len(reference_files) != 300 or len(generated_files) != 300:
    raise RuntimeError(
        f"數量錯誤：reference={len(reference_files)}, generated={len(generated_files)}"
    )

reference_ids = {path.stem for path in reference_files}
generated_ids = {path.stem for path in generated_files}
if reference_ids != generated_ids:
    raise RuntimeError("reference與generated的clip ID不一致")

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

reference_hashes = {sha256(path) for path in reference_files}
generated_hashes = {sha256(path) for path in generated_files}
overlap = reference_hashes & generated_hashes

print("Reference：", len(reference_files))
print("Generated：", len(generated_files))
print("相同內容雜湊：", len(overlap))
print("路徑是否相同：", REFERENCE_DIR.resolve() == GENERATED_DIR.resolve())

if overlap:
    raise RuntimeError("發現reference與generated內容相同，停止評估")
if REFERENCE_DIR.resolve() == GENERATED_DIR.resolve():
    raise RuntimeError("reference與generated指向同一資料夾")

print("評估防呆通過")


## 7. 計算CLAP（matched與shuffled sanity check）

這裡使用專案已保存的LAION-CLAP checkpoint，結果會記錄checkpoint名稱。Matched平均應明顯高於shuffled平均。


In [ ]:
import numpy as np
import torch
import laion_clap

CLAP_CKPT = target_weights / "music_speech_audioset_epoch_15_esc_89.98.pt"
if not CLAP_CKPT.exists():
    raise FileNotFoundError(CLAP_CKPT)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("CLAP裝置：", device)

clap_model = laion_clap.CLAP_Module(enable_fusion=False, amodel="HTSAT-base")
clap_model.load_ckpt(str(CLAP_CKPT), verbose=False)

caption_by_id = {row["id"]: row["caption"] for row in caption_rows}
ordered_files = [GENERATED_DIR / f"{clip_id}.wav" for clip_id in sorted(caption_by_id)]
ordered_captions = [caption_by_id[path.stem] for path in ordered_files]

audio_embeddings = []
text_embeddings = []
BATCH = 8

for start in range(0, len(ordered_files), BATCH):
    file_batch = [str(path) for path in ordered_files[start:start + BATCH]]
    text_batch = ordered_captions[start:start + BATCH]

    audio = clap_model.get_audio_embedding_from_filelist(
        x=file_batch,
        use_tensor=True,
    ).detach().cpu()
    text = clap_model.get_text_embedding(
        text_batch,
        use_tensor=True,
    ).detach().cpu()

    audio_embeddings.append(audio)
    text_embeddings.append(text)
    print(f"CLAP完成 {min(start + BATCH, len(ordered_files))}/300")

audio_embeddings = torch.cat(audio_embeddings, dim=0).float()
text_embeddings = torch.cat(text_embeddings, dim=0).float()
audio_embeddings = torch.nn.functional.normalize(audio_embeddings, dim=-1)
text_embeddings = torch.nn.functional.normalize(text_embeddings, dim=-1)

matched = (audio_embeddings * text_embeddings).sum(dim=-1).numpy()
rng = np.random.default_rng(20260817)
permutation = rng.permutation(len(text_embeddings))
shuffled = (audio_embeddings * text_embeddings[permutation]).sum(dim=-1).numpy()

clap_result = {
    "model": "fluxaudio_s_150k_stage6_last.pth",
    "clap_checkpoint": CLAP_CKPT.name,
    "count": int(len(matched)),
    "matched_mean": float(matched.mean()),
    "matched_std": float(matched.std()),
    "matched_median": float(np.median(matched)),
    "matched_min": float(matched.min()),
    "matched_max": float(matched.max()),
    "shuffled_mean": float(shuffled.mean()),
}

(RESULT_DIR / "clap_results.json").write_text(
    json.dumps(clap_result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(clap_result, ensure_ascii=False, indent=2))
if clap_result["matched_mean"] <= clap_result["shuffled_mean"]:
    raise RuntimeError("Matched CLAP未高於shuffled，請檢查manifest或評估流程")


## 8. 計算FAD（獨立reference與generated）

此格使用官方ICME評估repo的FAD程式。安裝評估依賴可能需要數分鐘；放在CLAP之後，避免套件問題影響已完成的生成與CLAP結果。


In [ ]:
EVAL_REPO = Path("/content/ICME26-ATTM-GC-Evaluation")

if not EVAL_REPO.exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/ntu-musicailab/ICME26-ATTM-GC-Evaluation.git",
            str(EVAL_REPO),
        ],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(EVAL_REPO / "requirements.txt")],
    check=True,
)

# Python 3.12 compatibility: older LAION-CLAP uses logging.info with
# multiple non-format arguments, which raises TypeError during checkpoint load.
import site
import py_compile

clap_hook = Path(site.getsitepackages()[0]) / "laion_clap" / "hook.py"
hook_lines = clap_hook.read_text(encoding="utf-8").splitlines()
patched = False
for index, line in enumerate(hook_lines):
    if "logging.info" in line and "Loaded" in line and "Unloaded" in line:
        indentation = line[:len(line) - len(line.lstrip())]
        hook_lines[index] = (
            indentation
            + 'logging.info(str(n) + "\\t" + ("Loaded" if n in ckpt else "Unloaded"))'
        )
        patched = True

if patched:
    clap_hook.write_text("\n".join(hook_lines) + "\n", encoding="utf-8")

py_compile.compile(str(clap_hook), doraise=True)
print("LAION-CLAP Python 3.12 logging修正：OK")

fad_command = [
    sys.executable,
    "src/fad.py",
    str(REFERENCE_DIR),
    str(GENERATED_DIR),
    "-w", "1",
]

fad_run = subprocess.run(
    fad_command,
    cwd=EVAL_REPO,
    capture_output=True,
    text=True,
)

(RESULT_DIR / "fad_stdout.txt").write_text(fad_run.stdout, encoding="utf-8")
(RESULT_DIR / "fad_stderr.txt").write_text(fad_run.stderr, encoding="utf-8")

print("FAD return code：", fad_run.returncode)
print(fad_run.stdout[-5000:])
if fad_run.returncode != 0:
    print(fad_run.stderr[-5000:])
    raise RuntimeError("FAD執行失敗；生成音訊與CLAP結果仍已安全保存在Drive")

result_folder = EVAL_REPO / "result"
for result_file in result_folder.glob("fad*.csv"):
    destination = RESULT_DIR / result_file.name
    shutil.copy2(result_file, destination)
    print("FAD結果：", destination)
    print(destination.read_text(encoding="utf-8"))


## 9. 完成與安全結束

Drive永久輸出位於：

```text
MyDrive/FluxAudio_evaluation/150k/
├── reference_300x10s/
├── generated_300x9.975s/
├── manifest_300.json
└── results/
```

確認`results/clap_results.json`與FAD CSV存在後，儲存notebook並選擇 **執行階段 → 中斷連線並刪除執行階段**。

KL／FD-PANN／FD-PaSST會在這套reference/generated防呆通過後另行加入，避免再次混用cache。
